# PCFM on GIFT-Eval

Zero-shot evaluation of [PCFM](https://github.com/software-competence-center-hagenberg/PCFM)
(weights: [Hugging Face](https://huggingface.co/software-competence-center-hagenberg/PCFM)) on the full GIFT-Eval suite
(97 configurations). The model code is `model.py` of the PCFM repository; the GIFT-Eval predictor and the evaluation
loop are in `pcfm_gifteval.py` next to it. Inference is training-free: a small test-time ensemble (sign flip x
dilation), short-context stretching and a context cap, all described at the top of `pcfm_gifteval.py`.

Runs on one GPU (Colab A100 / L4); the result is `results/PCFM/all_results.csv`.

## 1. Setup: PCFM, GIFT-Eval, data

In [ ]:
PCFM_REPO = "https://github.com/MMayr96/PCFM-Test.git"   # test repo; upstream: software-competence-center-hagenberg/PCFM
!git clone -q $PCFM_REPO PCFM
!git clone -q https://github.com/SalesforceAIResearch/gift-eval.git
# relax two pins for recent Python versions
!sed -i 's/scipy~=1.11.3/scipy>=1.11.3/g; s/numpy~=1.26.0/numpy>=1.26.0/g' gift-eval/pyproject.toml
!pip install -q -e gift-eval
!pip install -q -r PCFM/requirements.txt

import os
from huggingface_hub import snapshot_download
os.environ["GIFT_EVAL"] = os.path.abspath("gift-eval-data")
snapshot_download(repo_id="Salesforce/GiftEval", repo_type="dataset", local_dir=os.environ["GIFT_EVAL"], max_workers=16)

## 2. Evaluate

In [ ]:
import logging, sys
sys.path.insert(0, "PCFM")
from pcfm_gifteval import run_deploy_eval

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
logging.getLogger("gluonts.model.forecast").setLevel(logging.ERROR)

results = run_deploy_eval(properties="gift-eval/notebooks/dataset_properties.json",
                          out="results/PCFM/all_results.csv", model_name="PCFM")
results[["dataset", "eval_metrics/MASE[0.5]", "eval_metrics/mean_weighted_sum_quantile_loss"]]

## 3. Submission files

`config.json` next to the CSV, then the GIFT-Eval validator. Check `model_type` and `testdata_leakage` against the
pretraining corpus before submitting (see the GIFT-Eval README for the definitions).

In [ ]:
import json
json.dump({
    "model": "PCFM",
    "model_type": "pretrained",
    "model_dtype": "float32",
    "model_link": "https://huggingface.co/software-competence-center-hagenberg/PCFM",
    "code_link": "https://github.com/MMayr96/PCFM-Test/blob/main/gifteval.ipynb",
    "org": "Software Competence Center Hagenberg",
    "testdata_leakage": "No",
    "replication_code_available": "Yes",
}, open("results/PCFM/config.json", "w"), indent=4)
!cd gift-eval && python scripts/validate_results.py ../results/PCFM